In [1]:
import math
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader
from torchvision.datasets import ImageFolder
import torchvision.transforms.v2 as transforms
import timm

# Implementation of the core Additive Angular Margin Loss (ArcFace Module)
class ArcMarginProduct(nn.Module):
    def __init__(self, in_features, out_features, s=64.0, m=0.50):
        super(ArcMarginProduct, self).__init__()
        self.in_features = in_features
        self.out_features = out_features
        self.s = s
        self.m = m
        # Initialize class center weights layer
        self.weight = nn.Parameter(torch.FloatTensor(out_features, in_features))
        nn.init.xavier_uniform_(self.weight)

        self.cos_m = math.cos(m)
        self.sin_m = math.sin(m)
        self.th = math.cos(math.pi - m)
        self.mm = math.sin(math.pi - m) * m

    def forward(self, input, label):
        # 1. Cosine similarity calculation: L2 normalized weights and inputs
        cosine = F.linear(F.normalize(input), F.normalize(self.weight))
        sine = torch.sqrt(1.0 - torch.pow(cosine, 2)).clamp(0, 1)
        
        # 2. Apply trigonometric identity for cos(theta + m)
        phi = cosine * self.cos_m - sine * self.sin_m
        # Handle boundary conditions where theta + m > pi
        phi = torch.where(cosine > self.th, phi, cosine - self.mm)
        
        # 3. Transform targets using one-hot mapping masks
        one_hot = torch.zeros(cosine.size(), device=input.device)
        one_hot.scatter_(1, label.view(-1, 1).long(), 1)
        
        output = (one_hot * phi) + ((1.0 - one_hot) * cosine)
        output *= self.s
        return output

In [2]:
DATA_DIR = "./datasets/Retail-YU_reformed"
BATCH_SIZE = 64

# Basic normalization metrics inheriting ImageNet defaults
data_transforms = transforms.Compose([
    transforms.ToImage(),
    transforms.ToDtype(torch.float32, scale=True),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
])

train_dataset = ImageFolder(root=f"{DATA_DIR}/train", transform=data_transforms)
val_dataset = ImageFolder(root=f"{DATA_DIR}/val", transform=data_transforms)

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True, num_workers=4, pin_memory=True)
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=4, pin_memory=True)

NUM_CLASSES = len(train_dataset.classes)
print(f"Dataset configuration finalized. Tracked Classes Count: {NUM_CLASSES}")

Dataset configuration finalized. Tracked Classes Count: 1485


In [3]:
import os
import lightning as L
from lightning.pytorch.tuner import Tuner
from lightning.pytorch.callbacks import EarlyStopping, ModelCheckpoint
import torch
import torch.nn as nn
import torch.nn.functional as F
import timm

# 1. Bring over the ArcMarginProduct layer from your previous setup
class ArcMarginProduct(nn.Module):
    def __init__(self, in_features, out_features, s=64.0, m=0.50):
        super().__init__()
        self.in_features = in_features
        self.out_features = out_features
        self.s = s
        self.m = m
        self.weight = nn.Parameter(torch.FloatTensor(out_features, in_features))
        nn.init.xavier_uniform_(self.weight)
        
        self.cos_m = math.cos(m)
        self.sin_m = math.sin(m)
        self.th = math.cos(math.pi - m)
        self.mm = math.sin(math.pi - m) * m

    def forward(self, input, label):
        cosine = F.linear(F.normalize(input), F.normalize(self.weight))
        sine = torch.sqrt(1.0 - torch.pow(cosine, 2)).clamp(0, 1)
        phi = cosine * self.cos_m - sine * self.sin_m
        phi = torch.where(cosine > self.th, phi, cosine - self.mm)
        one_hot = torch.zeros(cosine.size(), device=input.device)
        one_hot.scatter_(1, label.view(-1, 1).long(), 1)
        output = (one_hot * phi) + ((1.0 - one_hot) * cosine)
        output *= self.s
        return output

# 2. Define the complete Lightning System
class ArcFaceVendingSystem(L.LightningModule):
    def __init__(self, num_classes, embedding_size=512, lr=1e-4):
        super().__init__()
        # Essential statement to log hyperparameter changes (required for LR Finder)
        self.save_hyperparameters()
        self.lr = lr
        
        # Identity-overwritten backbone initialization
        self.backbone = timm.create_model('hf_hub:timm/convnext_tiny.fb_in22k_ft_in1k', pretrained=True)
        num_features = self.backbone.num_features
        self.backbone.head.fc = nn.Identity()
        
        self.bn_layer = nn.BatchNorm1d(num_features)
        self.fc_embedding = nn.Linear(num_features, embedding_size)
        self.arcface_head = ArcMarginProduct(in_features=embedding_size, out_features=num_classes)
        self.criterion = nn.CrossEntropyLoss()

    def extract_embeddings(self, x):
        features = self.backbone(x)
        features = self.bn_layer(features)
        embeddings = self.fc_embedding(features)
        return F.normalize(embeddings)

    def training_step(self, batch, batch_idx):
        images, labels = batch
        features = self.backbone(images)
        features = self.bn_layer(features)
        embeddings = self.fc_embedding(features)
        
        logits = self.arcface_head(embeddings, labels)
        loss = self.criterion(logits, labels)
        
        # Calculate metric monitoring
        preds = torch.argmax(logits, dim=1)
        acc = (preds == labels).float().mean()
        
        # Log values to show them automatically inside the progress bar
        self.log("train_loss", loss, prog_bar=True, on_step=False, on_epoch=True)
        self.log("train_acc", acc, prog_bar=True, on_step=False, on_epoch=True)
        return loss

    def validation_step(self, batch, batch_idx):
        images, labels = batch
        features = self.backbone(images)
        features = self.bn_layer(features)
        embeddings = self.fc_embedding(features)
        
        logits = self.arcface_head(embeddings, labels)
        loss = self.criterion(logits, labels)
        
        preds = torch.argmax(logits, dim=1)
        acc = (preds == labels).float().mean()
        
        self.log("val_loss", loss, prog_bar=True, on_step=False, on_epoch=True)
        self.log("val_acc", acc, prog_bar=True, on_step=False, on_epoch=True)

    def configure_optimizers(self):
        # Always reference self.lr so that the Tuner can rewrite it dynamically
        return torch.optim.AdamW(self.parameters(), lr=self.lr, weight_decay=1e-2)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
# Create instance targeting your class count variable
model_system = ArcFaceVendingSystem(num_classes=NUM_CLASSES)

# Optionally resume from a previously fine-tuned checkpoint instead of starting
# purely from the ImageNet-pretrained backbone. Tensors whose shape no longer
# matches (e.g. arcface_head.weight if NUM_CLASSES changed) are skipped and
# left freshly initialized; backbone/bn_layer/fc_embedding are warm-started.
RESUME_FROM = "./models/tuned/items_classification_convnext_tiny.fb_in22k_ft_in1k.pt"
if os.path.exists(RESUME_FROM):
    prev_state = torch.load(RESUME_FROM, map_location="cpu", weights_only=False)
    own_state  = model_system.state_dict()
    compatible = {k: v for k, v in prev_state.items()
                   if k in own_state and v.shape == own_state[k].shape}
    skipped = sorted(set(prev_state) - set(compatible))
    model_system.load_state_dict(compatible, strict=False)
    print(f"Resumed {len(compatible)}/{len(prev_state)} tensors from {RESUME_FROM}")
    if skipped:
        print(f"  skipped (shape mismatch / not in model): {skipped}")
else:
    print("No existing checkpoint found, starting from ImageNet-pretrained weights")

# Save the best epoch (by val_loss) and stop early once it stops improving,
# so max_epochs only acts as a ceiling rather than a fixed budget.
checkpoint_callback = ModelCheckpoint(
    dirpath="./models/tuned/checkpoints",
    filename="item_classifier-{epoch}-{val_loss:.3f}",
    monitor="val_loss",
    mode="min",
    save_top_k=1,
)
early_stop_callback = EarlyStopping(monitor="val_loss", mode="min", patience=3)

# Instantiate basic multi-hardware execution profile runner
trainer = L.Trainer(
    max_epochs=30,
    accelerator="auto", # Automatically detects and targets CUDA if available
    devices=1,
    enable_progress_bar=True,
    callbacks=[checkpoint_callback, early_stop_callback],
)

# Initialize the automated Tuning engine
tuner = Tuner(trainer)

torch.set_float32_matmul_precision('medium')

print("Starting automatic learning rate range test exploration...")
# early_stop_threshold=None disables the "diverging loss" early stop. With a
# resumed/fine-tuned checkpoint the starting loss is already low, so the
# default threshold (loss > 4x best_loss) triggers within a handful of steps
# and leaves too few points (suggestion() needs >10) to fit a curve. max_lr
# is also narrowed since the default of 1.0 is wildly unstable for fine-tuning.
lr_finder = tuner.lr_find(
    model_system,
    train_dataloaders=train_loader,
    early_stop_threshold=None,
    max_lr=1e-2,
)

# Pick out the suggested optimization value automatically, falling back to the
# current lr if the curve still doesn't yield a clear suggestion
new_lr = lr_finder.suggestion()
if new_lr is None:
    print(f"No LR suggestion found, keeping current rate: {model_system.lr:.6f}")
else:
    print(f"Optimal inflection point identified! Setting learning rate to: {new_lr:.6f}")
    model_system.lr = new_lr

# Optional: Render the traditional LR Loss-Slope breakdown graph
# fig = lr_finder.plot(suggest=True)
# fig.show()

GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning Experiments platform.


Resumed 188/188 tensors from ./models/tuned/items_classification_convnext_tiny.fb_in22k_ft_in1k.pt
Starting automatic learning rate range test exploration...


/home/potata/.pyenv/versions/3.12.11/lib/python3.12/site-packages/lightning/pytorch/trainer/configuration_validator.py:70: You defined a `validation_step` but have no `val_dataloader`. Skipping val loop.
LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0,1]
`weights_only` was not set, defaulting to `False`.


Finding best initial lr:   0%|          | 0/100 [00:00<?, ?it/s]

`Trainer.fit` stopped: `max_steps=100` reached.
Restoring states from the checkpoint path at /home/potata/Desktop/repi/vendingMachineParser/.lr_find_1880c62a-79ed-476a-b15d-59be43fcaa9f.ckpt
Restored all states from the checkpoint at /home/potata/Desktop/repi/vendingMachineParser/.lr_find_1880c62a-79ed-476a-b15d-59be43fcaa9f.ckpt
Learning rate set to 9.120108393559099e-08


Optimal inflection point identified! Setting learning rate to: 0.000000


In [ ]:
model_system.lr = 1e-05
trainer.fit(model_system, train_dataloaders=train_loader, val_dataloaders=val_loader)

LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0,1]


┏━━━┳━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━┳━━━━━━━━┳━━━━━━━┳━━━━━━━┓
┃   ┃ Name         ┃ Type             ┃ Params ┃ Mode  ┃ FLOPs ┃
┡━━━╇━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━╇━━━━━━━━╇━━━━━━━╇━━━━━━━┩
│ 0 │ backbone     │ ConvNeXt         │ 27.8 M │ train │     0 │
│ 1 │ bn_layer     │ BatchNorm1d      │  1.5 K │ train │     0 │
│ 2 │ fc_embedding │ Linear           │  393 K │ train │     0 │
│ 3 │ arcface_head │ ArcMarginProduct │  760 K │ train │     0 │
│ 4 │ criterion    │ CrossEntropyLoss │      0 │ train │     0 │
└───┴──────────────┴──────────────────┴────────┴───────┴───────┘

Trainable params: 29.0 M                                                                                           
Non-trainable params: 0                                                                                            
Total params: 29.0 M                                                                                               
Total estimated model params size (MB): 115.903                                                                    
Modules in train mode: 216                                                                                         
Modules in eval mode: 0                                                                                            
Total FLOPs: 0

Output()


Detected KeyboardInterrupt, attempting graceful shutdown ...


SystemExit: 1

/home/potata/.pyenv/versions/3.12.11/lib/python3.12/site-packages/IPython/core/interactiveshell.py:3707: UserWarning: To exit: use 'exit', 'quit', or Ctrl-D.
  warn("To exit: use 'exit', 'quit', or Ctrl-D.", stacklevel=1)


In [ ]:
#save model
import torch

# Restore the best checkpoint (lowest val_loss) before exporting — with early
# stopping, the in-memory weights at the end of fit() may be from a worse,
# later epoch than the one ModelCheckpoint kept.
best_path = checkpoint_callback.best_model_path
if best_path:
    print(f"Loading best checkpoint: {best_path}")
    best_model = ArcFaceVendingSystem.load_from_checkpoint(best_path, map_location="cpu")
else:
    print("No checkpoint recorded, saving current weights")
    best_model = model_system

# Extract the pure PyTorch state dictionary from the Lightning system
torch.save(best_model.state_dict(), "./models/tuned/items_classification_convnext_tiny.fb_in22k_ft_in1k.pt")
print("Pure PyTorch weights saved cleanly.")

In [ ]:
import os
import numpy as np
from torchvision.datasets import ImageFolder
from torch.utils.data import DataLoader

# 1. Double check and dynamically filter out any empty folders right before loading
gallery_dir = f"{DATA_DIR}/gallery"
for folder in os.listdir(gallery_dir):
    f_path = os.path.join(gallery_dir, folder)
    if os.path.isdir(f_path) and not os.listdir(f_path):
        os.rmdir(f_path) # Clear it out so ImageFolder doesn't throw a fit

# 2. Load your custom gallery dataset safely
gallery_dataset = ImageFolder(root=gallery_dir, transform=data_transforms)
gallery_loader = DataLoader(gallery_dataset, batch_size=1, shuffle=False)

# 3. Put the Lightning system into evaluation mode
model_system.eval()
current_device = model_system.device
gallery_database = {}

print(f"Generating target embedding registry using device: {current_device}...")

with torch.no_grad():
    for img, label_idx in gallery_loader:
        img = img.to(current_device)
        
        # Extract the normalized 512-dim embedding vector
        embedding = model_system.extract_embeddings(img).cpu().numpy().flatten()
        
        class_name = gallery_dataset.classes[label_idx]
        
        if class_name not in gallery_database:
            gallery_database[class_name] = []
        gallery_database[class_name].append(embedding)

# 4. Average out embeddings if a class has multiple references
for class_name in gallery_database:
    gallery_database[class_name] = np.mean(gallery_database[class_name], axis=0)

# 5. Save your reference dictionary to disk
os.makedirs("./models/tuned", exist_ok=True)
np.save("./models/tuned/items_classification.npy", gallery_database, allow_pickle=True)
print(f"Successfully registered {len(gallery_database)} unique product reference keys.")

In [ ]:
# --- Sanity check: gallery embedding distribution & class separability ---
import numpy as np
import matplotlib.pyplot as plt
from sklearn.decomposition import PCA
from sklearn.manifold import TSNE

model_system.eval()

# recompute per-image (non-averaged) embeddings so we can see the spread within each class
embeddings, labels = [], []
with torch.no_grad():
    for img, label_idx in gallery_loader:
        img = img.to(current_device)
        emb = model_system.extract_embeddings(img).cpu().numpy().flatten()
        embeddings.append(emb)
        labels.append(gallery_dataset.classes[label_idx])

embeddings  = np.stack(embeddings)            # (N, 512), L2-normalized
labels      = np.array(labels)
class_names = sorted(set(labels))

# numeric sanity: a healthy embedding space has intra-class similarity >> inter-class
sim = embeddings @ embeddings.T
intra, inter = [], []
for i in range(len(labels)):
    for j in range(i + 1, len(labels)):
        (intra if labels[i] == labels[j] else inter).append(sim[i, j])

print(f"mean intra-class cosine similarity: {np.mean(intra):.3f}")
print(f"mean inter-class cosine similarity: {np.mean(inter):.3f}")

# reduce 512-d embeddings to 3D (PCA + t-SNE, the same methods TensorBoard's
# Embedding Projector offers) and plot, colored by class
n          = len(embeddings)
perplexity = max(2, min(30, n - 1))
reductions = {
    "PCA":   PCA(n_components=3).fit_transform(embeddings),
    "t-SNE": TSNE(n_components=3, perplexity=perplexity, init="pca", random_state=0).fit_transform(embeddings),
}

cmap     = plt.get_cmap("tab20", len(class_names))
color_of = {name: cmap(i) for i, name in enumerate(class_names)}

fig = plt.figure(figsize=(14, 7))
for i, (method, reduced) in enumerate(reductions.items(), start=1):
    ax = fig.add_subplot(1, 2, i, projection="3d")
    for name in class_names:
        mask = labels == name
        ax.scatter(reduced[mask, 0], reduced[mask, 1], reduced[mask, 2],
                   color=color_of[name], label=name, s=40)
    ax.set_title(f"Gallery embeddings — {method} (3D)")
    ax.legend(fontsize=7, loc="upper left", bbox_to_anchor=(1.05, 1))

plt.tight_layout()
plt.show()

In [ ]:
def classify_product_crop(image_path, model, reference_gallery, transform):
    # 1. Load and prep the image (assumes it was already square-padded via split script)
    from PIL import Image
    img = Image.open(image_path)
    img_tensor = transform(img).unsqueeze(0).to(device)
    
    # 2. Extract embedding
    model.eval()
    with torch.no_grad():
        query_embedding = model.extract_embeddings(img_tensor).cpu().numpy().flatten()
    
    # 3. Compute cosine similarity against all gallery items
    best_match = None
    highest_score = -1.0
    
    for class_name, ref_embedding in reference_gallery.items():
        # Cosine similarity formula for normalized vectors: dot product
        similarity = np.dot(query_embedding, ref_embedding)
        if similarity > highest_score:
            highest_score = similarity
            best_match = class_name
            
    return best_match, highest_score

# Usage example:
# match, score = classify_product_crop("path_to_vending_crop.jpg", model, gallery_database, data_transforms)
# print(f"Identified product: {match} with confidence score: {score:.4f}")